# Ribosome-Cascade: Scale-Up Experiments (Colab A100)
## Validating the "lighter model on metatokens" thesis at 250M+ params

**What we proved locally:**  
A 49M ribosome-tiny model (2 embed + 4 upper layers, 16 metatokens) beats a 63M standard transformer (12 layers, 256 raw tokens) by 22× in perplexity on OpenWebText.

**What this notebook tests:**  
1. Does the advantage hold at 250M params? (hidden=1024, more layers)  
2. Does the advantage grow at longer context? (1024 tokens → 64 metatokens)  
3. Does lower CE translate to downstream task accuracy? (HellaSwag, LAMBADA)


In [ ]:
# Setup
!pip install -q transformers datasets accelerate

import torch
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


In [ ]:
# Clone repo
!git clone https://github.com/J-Lehrer/Ribosome-Cascade.git
%cd Ribosome-Cascade


## 1. Architecture at 250M scale
Scaling up from hidden=512 to hidden=1024, increasing layer counts.


In [ ]:
import sys
sys.path.insert(0, "/content/Ribosome-Cascade")
from native_arch_v1 import *
from exp2_lighter import BigBaseline, RibosomeTiny

# 250M-class models
big = BigBaseline(
    vocab_size=50257, hidden_size=1024, n_heads=16,
    n_layers=16, max_seq_len=1024
)
print(f"Big baseline: {big.count_params():,} params")

tiny = RibosomeTiny(
    vocab_size=50257, hidden_size=1024, n_heads=16,
    embed_layers=4, upper_layers=6,
    max_seq_len=1024, n_chunks=64
)
print(f"Ribosome tiny: {tiny.count_params():,} params")

# Smoke test
x = torch.randint(0, 50257, (1, 256))
loss_b, _ = big(x, x)
loss_t, _, imp = tiny(x, x)
print(f"Big loss: {loss_b.item():.2f}, Tiny loss: {loss_t.item():.2f}")
print(f"Importance range: [{imp.min().item():.3f}, {imp.max().item():.3f}]")

del big, tiny
torch.cuda.empty_cache()


## 2. Train both at 250M scale on OpenWebText
Training for 50K steps each (~200M tokens).  
Expected runtime: ~2 hours per model on A100.


In [ ]:
from exp2_lighter import train_model, BigBaseline, RibosomeTiny
from transformers import AutoTokenizer
import os

device = torch.device("cuda")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

class Args:
    device = "cuda"
    max_vram_gb = 35.0
    max_length = 1024
    batch_size = 4
    grad_accum = 8  # effective batch = 32
    epochs = 1
    steps_per_epoch = 50000
    max_lr = 3e-4
    min_lr = 3e-5
    log_every = 200
    eval_every = 5000
    dataset = "openwebtext"
    output_dir = "/content/scale_experiment"

args = Args()
os.makedirs(args.output_dir, exist_ok=True)

# Apply VRAM cap
total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
frac = min(args.max_vram_gb / total_vram, 0.95)
torch.cuda.set_per_process_memory_fraction(frac)
print(f"VRAM cap: {args.max_vram_gb}GB ({frac:.0%} of {total_vram:.0f}GB)")


In [ ]:
# Train big baseline (16 layers, 1024 tokens)
big = BigBaseline(
    vocab_size=len(tokenizer), hidden_size=1024, n_heads=16,
    n_layers=16, max_seq_len=args.max_length
).to(device)
print(f"Big: {big.count_params():,} params")

big_val = train_model(big, "big_16L_1024h", tokenizer, device, args)
print(f"\nBig baseline final val CE: {big_val:.4f}")
del big
torch.cuda.empty_cache()


In [ ]:
# Train ribosome tiny (4 embed + 6 upper, 64 metatokens)
tiny = RibosomeTiny(
    vocab_size=len(tokenizer), hidden_size=1024, n_heads=16,
    embed_layers=4, upper_layers=6,
    max_seq_len=args.max_length, n_chunks=64
).to(device)
print(f"Tiny: {tiny.count_params():,} params")

tiny_val = train_model(tiny, "ribosome_tiny_1024h", tokenizer, device, args,
                        is_ribosome=True)
print(f"\nRibosome tiny final val CE: {tiny_val:.4f}")
del tiny
torch.cuda.empty_cache()


In [ ]:
# Results comparison
print("=" * 60)
print("250M SCALE RESULTS")
print("=" * 60)
print(f"Big baseline (16L, 1024 tokens): val CE = {big_val:.4f}  ppl = {2.718**big_val:.0f}")
print(f"Ribosome tiny (10L, 64 chunks):  val CE = {tiny_val:.4f}  ppl = {2.718**tiny_val:.0f}")
print(f"Delta: {tiny_val - big_val:+.4f} CE")
print(f"Perplexity ratio: {2.718**big_val / 2.718**tiny_val:.1f}x")


## 3. Downstream evaluation
Testing on HellaSwag (commonsense reasoning) and LAMBADA (long-range prediction).


In [ ]:
# HellaSwag evaluation
from datasets import load_dataset

def eval_hellaswag(model, tokenizer, device, n_samples=500, is_ribosome=False):
    ds = load_dataset("Rowan/hellaswag", split="validation")
    correct = 0
    total = 0

    model.eval()
    with torch.no_grad():
        for i, ex in enumerate(ds):
            if i >= n_samples:
                break
            ctx = ex["ctx"]
            endings = ex["endings"]
            label = int(ex["label"])

            losses = []
            for ending in endings:
                text = ctx + " " + ending
                ids = tokenizer.encode(text, truncation=True, max_length=256)
                input_ids = torch.tensor([ids[:-1]]).to(device)
                labels = torch.tensor([ids[1:]]).to(device)

                if is_ribosome:
                    loss, _, _ = model(input_ids, labels)
                else:
                    loss, _ = model(input_ids, labels)
                losses.append(loss.item())

            pred = losses.index(min(losses))
            if pred == label:
                correct += 1
            total += 1

    return correct / total

print("Evaluating on HellaSwag...")

# Load best checkpoints
big = BigBaseline(vocab_size=len(tokenizer), hidden_size=1024, n_heads=16,
                  n_layers=16, max_seq_len=1024).to(device)
ckpt = torch.load(f"{args.output_dir}/big_16L_1024h/best.pt", map_location=device, weights_only=False)
big.load_state_dict(ckpt["model"])

tiny = RibosomeTiny(vocab_size=len(tokenizer), hidden_size=1024, n_heads=16,
                    embed_layers=4, upper_layers=6, max_seq_len=1024, n_chunks=64).to(device)
ckpt = torch.load(f"{args.output_dir}/ribosome_tiny_1024h/best.pt", map_location=device, weights_only=False)
tiny.load_state_dict(ckpt["model"])

big_acc = eval_hellaswag(big, tokenizer, device, n_samples=500)
print(f"Big baseline HellaSwag: {big_acc*100:.1f}%")
del big; torch.cuda.empty_cache()

tiny_acc = eval_hellaswag(tiny, tokenizer, device, n_samples=500, is_ribosome=True)
print(f"Ribosome tiny HellaSwag: {tiny_acc*100:.1f}%")
del tiny; torch.cuda.empty_cache()

print(f"\nDelta: {(tiny_acc-big_acc)*100:+.1f}%")


In [ ]:
# LAMBADA evaluation (next word prediction accuracy)
def eval_lambada(model, tokenizer, device, n_samples=500, is_ribosome=False):
    ds = load_dataset("EleutherAI/lambada_openai", "en", split="test")
    correct = 0
    total = 0

    model.eval()
    with torch.no_grad():
        for i, ex in enumerate(ds):
            if i >= n_samples:
                break
            text = ex["text"]
            ids = tokenizer.encode(text, truncation=True, max_length=256)
            if len(ids) < 3:
                continue

            input_ids = torch.tensor([ids[:-1]]).to(device)
            target = ids[-1]

            if is_ribosome:
                _, logits, _ = model(input_ids)
            else:
                _, logits = model(input_ids)

            pred = logits[0, -1].argmax().item()
            if pred == target:
                correct += 1
            total += 1

    return correct / max(total, 1)

print("Evaluating on LAMBADA...")

big = BigBaseline(vocab_size=len(tokenizer), hidden_size=1024, n_heads=16,
                  n_layers=16, max_seq_len=1024).to(device)
ckpt = torch.load(f"{args.output_dir}/big_16L_1024h/best.pt", map_location=device, weights_only=False)
big.load_state_dict(ckpt["model"])

tiny = RibosomeTiny(vocab_size=len(tokenizer), hidden_size=1024, n_heads=16,
                    embed_layers=4, upper_layers=6, max_seq_len=1024, n_chunks=64).to(device)
ckpt = torch.load(f"{args.output_dir}/ribosome_tiny_1024h/best.pt", map_location=device, weights_only=False)
tiny.load_state_dict(ckpt["model"])

big_lam = eval_lambada(big, tokenizer, device, n_samples=500)
print(f"Big baseline LAMBADA: {big_lam*100:.1f}%")
del big; torch.cuda.empty_cache()

tiny_lam = eval_lambada(tiny, tokenizer, device, n_samples=500, is_ribosome=True)
print(f"Ribosome tiny LAMBADA: {tiny_lam*100:.1f}%")
del tiny; torch.cuda.empty_cache()

print(f"\nDelta: {(tiny_lam-big_lam)*100:+.1f}%")


## 4. Final summary


In [ ]:
import math

print("=" * 70)
print("RIBOSOME-CASCADE SCALE-UP RESULTS")
print("=" * 70)
print(f"")
print(f"{'Metric':<30s} {'Big 16L':>12s} {'Ribo Tiny':>12s} {'Delta':>12s}")
print(f"{'-'*66}")
print(f"{'Val CE':<30s} {big_val:>12.4f} {tiny_val:>12.4f} {tiny_val-big_val:>+12.4f}")
print(f"{'Perplexity':<30s} {math.exp(big_val):>12.0f} {math.exp(tiny_val):>12.0f} {math.exp(big_val)/math.exp(tiny_val):>11.1f}x")
print(f"{'HellaSwag accuracy':<30s} {big_acc*100:>11.1f}% {tiny_acc*100:>11.1f}% {(tiny_acc-big_acc)*100:>+11.1f}%")
print(f"{'LAMBADA accuracy':<30s} {big_lam*100:>11.1f}% {tiny_lam*100:>11.1f}% {(tiny_lam-big_lam)*100:>+11.1f}%")
print(f"")
print(f"Big: ~250M params, 16 layers, processes 1024 raw tokens")
print(f"Tiny: ~250M params, 10 layers, processes 64 metatokens (16:1 compression)")

# Save results
import json
results = {
    "big_val_ce": big_val, "tiny_val_ce": tiny_val,
    "big_hellaswag": big_acc, "tiny_hellaswag": tiny_acc,
    "big_lambada": big_lam, "tiny_lambada": tiny_lam,
}
with open("/content/scale_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nResults saved to /content/scale_results.json")
print("Copy this file to your local machine before the runtime disconnects!")
